# Dataset Splitting & Configuration

In [ ]:
import os
import random
import shutil
import yaml

# Define paths
RAW_DIR = "../data/raw"
LABEL_DIR = "../data/labels"
PROCESSED_DIR = "../data/processed"

# YOLO requires a specific folder structure
FOLDERS = [
    "images/train", "images/val",
    "labels/train", "labels/val"
]

for folder in FOLDERS:
    os.makedirs(os.path.join(PROCESSED_DIR, folder), exist_ok=True)

# Get all images and their corresponding labels
all_images = sorted([f for f in os.listdir(RAW_DIR) if f.endswith('.jpg')])
data_pairs = []

for img in all_images:
    label = img.replace(".jpg", ".txt")
    if os.path.exists(os.path.join(LABEL_DIR, label)):
        data_pairs.append((img, label))

# Shuffle and split (80% Train, 20% Val)
random.seed(42) # For reproducibility
random.shuffle(data_pairs)

split_idx = int(len(data_pairs) * 0.8)
train_pairs = data_pairs[:split_idx]
val_pairs = data_pairs[split_idx:]

def copy_data(pairs, split_name):
    for img, label in pairs:
        # Copy image
        shutil.copy(
            os.path.join(RAW_DIR, img),
            os.path.join(PROCESSED_DIR, f"images/{split_name}", img)
        )
        # Copy label
        shutil.copy(
            os.path.join(LABEL_DIR, label),
            os.path.join(PROCESSED_DIR, f"labels/{split_name}", label)
        )

# Execute the copy
print("Splitting dataset...")
copy_data(train_pairs, "train")
copy_data(val_pairs, "val")

print(f"Training set: {len(train_pairs)} images")
print(f"Validation set: {len(val_pairs)} images")

yaml_path = "../data/dataset.yaml"

abs_path = os.path.abspath(PROCESSED_DIR)

yaml_content = {
    "path": abs_path,
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "worker"
    }
}

with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)

print(f"Created YOLO configuration file at '{yaml_path}'")

Splitting dataset...
Training set: 60 images
Validation set: 16 images
Created YOLO configuration file at '../data/dataset.yaml'
